# 第 39 章：Capstone 项目工作流、项目模板与可信交付

这个 notebook 用一个极小的 project board，对比“只看模型兴奋度”的 naive baseline 和“先过科学问题 / 数据边界 / 验证 / 文献 / AI-use statement gate”的更可靠 capstone workflow。


In [ ]:
from __future__ import annotations

import csv
import platform
from collections import Counter
from pathlib import Path

DATA_PATH = Path('../../data/small/capstone_project_workflow_demo.csv')


def load_projects(path):
    rows = []
    with path.open(encoding='utf-8', newline='') as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            row['model_hype_score'] = float(row['model_hype_score'])
            rows.append(row)
    return rows


def ordered_counts(rows, field):
    counts = Counter(row[field] for row in rows)
    return {key: counts[key] for key in sorted(counts)}


rows = load_projects(DATA_PATH)
print(f'Loaded {len(rows)} project cards from {DATA_PATH.name}')
print('Python version:', platform.python_version())
print('Science-question status:', ordered_counts(rows, 'science_question_status'))
print('Data-license status:', ordered_counts(rows, 'data_license_status'))
print('Sharing boundary:', ordered_counts(rows, 'sharing_boundary'))
print('Reference route:', ordered_counts(rows, 'reference_route'))


In [ ]:
reference_ready_ids = sorted(
    row['project_id']
    for row in rows
    if row['reference_route'] == 'ready_for_capstone'
)
top3_hype = sorted(
    rows,
    key=lambda row: row['model_hype_score'],
    reverse=True,
)[:3]
baseline_ready_hits = sum(
    row['reference_route'] == 'ready_for_capstone'
    for row in top3_hype
)
baseline_ready_precision = baseline_ready_hits / len(top3_hype)

print('Reference ready projects:', reference_ready_ids)
print('Top-3 model-hype projects:')
for row in top3_hype:
    print(
        f"  {row['project_id']}: hype={row['model_hype_score']:.2f}, "
        f"route={row['reference_route']}, theme={row['theme']}"
    )
print('Baseline ready precision:', round(baseline_ready_precision, 3))


In [ ]:
def route_project(row):
    if row['data_license_status'] != 'clear':
        return 'blocked', 'data license is not yet clear'
    if row['sharing_boundary'] != 'local_only':
        return 'blocked', 'project materials should not be pasted into an unknown external service'
    if row['science_question_status'] != 'clear':
        return 'revise_plan', 'science question is still too broad for a capstone-sized deliverable'
    if row['baseline_status'] != 'ready':
        return 'revise_plan', 'baseline is missing, so later model comparisons would be ungrounded'
    if row['validation_status'] != 'ready':
        return 'manual_review', 'validation and regression evidence still need a human signoff pass'
    if row['literature_ledger_status'] != 'ready':
        return 'manual_review', 'literature claims and caveats still need to be organized into a ledger'
    if row['ai_use_statement_status'] != 'drafted':
        return 'manual_review', 'AI-use statement is not yet aligned with the actual workflow log'
    if row['report_outline_status'] != 'ready':
        return 'manual_review', 'report outline is not yet ready for a reproducible handoff'
    return 'ready_for_capstone', 'all core gates are satisfied'


routed_rows = []
for row in rows:
    route, reason = route_project(row)
    routed = dict(row)
    routed['route'] = route
    routed['route_reason'] = reason
    routed_rows.append(routed)

exact_match_count = sum(
    row['route'] == row['reference_route']
    for row in routed_rows
)
workflow_accuracy = exact_match_count / len(routed_rows)

print('Workflow routing decisions:')
for row in routed_rows:
    print(
        f"  {row['project_id']}: predicted={row['route']}, "
        f"reference={row['reference_route']}, reason={row['route_reason']}"
    )
print('Exact routing accuracy:', round(workflow_accuracy, 3))


In [ ]:
ready_queue = [row for row in routed_rows if row['route'] == 'ready_for_capstone']
review_queue = [row for row in routed_rows if row['route'] == 'manual_review']
revise_queue = [row for row in routed_rows if row['route'] == 'revise_plan']
blocked_queue = [row for row in routed_rows if row['route'] == 'blocked']

ready_precision = sum(
    row['reference_route'] == 'ready_for_capstone'
    for row in ready_queue
) / max(len(ready_queue), 1)
ready_recall = sum(
    row['route'] == 'ready_for_capstone' and row['reference_route'] == 'ready_for_capstone'
    for row in routed_rows
) / max(len(reference_ready_ids), 1)

print('Ready queue:', [row['project_id'] for row in ready_queue])
print('Manual-review queue:', [row['project_id'] for row in review_queue])
print('Revise-plan queue:', [row['project_id'] for row in revise_queue])
print('Blocked queue:', [row['project_id'] for row in blocked_queue])
print('Structured-workflow ready precision:', round(ready_precision, 3))
print('Structured-workflow ready recall:', round(ready_recall, 3))


In [ ]:
def next_steps(row):
    steps = []
    if row['data_license_status'] != 'clear':
        steps.append('补一张数据来源与许可证卡，明确哪些材料不能进入公开仓库或外部服务。')
    if row['sharing_boundary'] != 'local_only':
        steps.append('停止把项目材料发送到共享边界不清楚的外部服务，并回写一条 usage log。')
    if row['science_question_status'] != 'clear':
        steps.append('把科学问题收窄成一个可测量、可展示的核心问题。')
    if row['baseline_status'] != 'ready':
        steps.append('先完成一个简单 baseline，再谈更复杂模型的增益。')
    if row['validation_status'] != 'ready':
        steps.append('补上 regression checks、关键数值 sanity check 和 clean rerun 记录。')
    if row['literature_ledger_status'] != 'ready':
        steps.append('把 related work、核心 claim 和 caveat 整理成可回查的 literature ledger。')
    if row['ai_use_statement_status'] != 'drafted':
        steps.append('根据真实 workflow 起草 AI-use statement，而不是最后再凭印象补一句。')
    if row['report_outline_status'] != 'ready':
        steps.append('把报告结构先写成 outline，确保图表、方法、结果和局限已经有位置。')
    return steps


def capstone_template(row):
    return {
        'science_question': row['theme'],
        'data_and_license': row['data_license_status'],
        'baseline': row['baseline_status'],
        'validation': row['validation_status'],
        'literature_ledger': row['literature_ledger_status'],
        'ai_use_statement': row['ai_use_statement_status'],
        'report_outline': row['report_outline_status'],
    }


print('Next-step checklist for non-ready projects:')
for row in routed_rows:
    if row['route'] == 'ready_for_capstone':
        continue
    print(f"\n{row['project_id']} -> {row['route']}")
    for step in next_steps(row):
        print(' -', step)

example_template = capstone_template(ready_queue[0])
print('\nTemplate snapshot for the first ready project:')
for key, value in example_template.items():
    print(f'  {key}: {value}')


In [ ]:
capstone_assistant_prompt = '''你是我的 capstone 项目助手。

任务：
1. 先阅读 project board，而不是先推荐模型；
2. 依次检查 science question、data/license、baseline、validation、
   literature ledger、AI-use statement 和 report outline；
3. 把每个项目 route 到 ready_for_capstone、manual_review、
   revise_plan 或 blocked；
4. 只有在 route 明确后，才给 next-step checklist。

输出格式：
- Ready projects
- Manual-review queue
- Revise-plan queue
- Blocked projects
- Next-step checklist by project
'''
print(capstone_assistant_prompt)


In [ ]:
workflow_checklist = [
    '先问科学问题和数据边界，而不是先问模型够不够新。',
    '没有 baseline 的项目，不应该直接跳到更复杂模型。',
    'validation、literature ledger 和 AI-use statement 都要留下可检查痕迹。',
    'manual_review 是正确出口，不是项目失败。',
    '最终展示前，要把 notebook、报告、图表和 Git 记录接成同一份交付证据。',
]

project_snapshot = {
    'dataset': DATA_PATH.name,
    'n_project_cards': len(rows),
    'baseline_top3_ready_precision': round(baseline_ready_precision, 3),
    'workflow_route_accuracy': round(workflow_accuracy, 3),
    'ready_queue': [row['project_id'] for row in ready_queue],
    'manual_review_queue': [row['project_id'] for row in review_queue],
    'blocked_queue': [row['project_id'] for row in blocked_queue],
    'python_version': platform.python_version(),
}

print('Workflow checklist:')
for item in workflow_checklist:
    print(' -', item)

print('\nProject snapshot:')
for key, value in project_snapshot.items():
    print(f'  {key}: {value}')
